# Garbage Classification — Training
CSE 4830 Group 9

Trains CustomCNN (baseline) and MobileNetV2 (transfer learning) and saves checkpoints + metrics.

In [ ]:
# ── Step 1: Clone repo ────────────────────────────────────────────────────
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/Garbage-CNN')
if not REPO_DIR.exists():
    os.system('git clone https://github.com/LevinMathew1/Garbage-CNN.git /content/Garbage-CNN')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Working directory:', os.getcwd())

In [ ]:
# ── Step 2: Install missing packages ─────────────────────────────────────
import os
os.system('pip install -q timm')
print('Done.')

In [ ]:
# ── Step 3: Verify GPU ────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 4: Mount Drive and extract data ─────────────────────────────────
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ZIP_SRC     = Path('/content/drive/MyDrive/archive.zip')
EXTRACT_DIR = Path('/content/data')

if not ZIP_SRC.exists():
    raise FileNotFoundError('archive.zip not found in Google Drive root. Upload it first.')

if not EXTRACT_DIR.exists():
    print('Extracting...')
    with zipfile.ZipFile(ZIP_SRC) as zf:
        zf.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Already extracted.')

print('Contents:', [p.name for p in sorted(EXTRACT_DIR.iterdir())])

In [ ]:
# ── Step 5: Resolve DATA_DIR ──────────────────────────────────────────────
from pathlib import Path

EXTRACT_DIR = Path('/content/data')
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def find_data_dir(root: Path) -> Path:
    for candidate in [root] + sorted([p for p in root.iterdir() if p.is_dir()]):
        subdirs = sorted([p for p in candidate.iterdir() if p.is_dir()])
        if not subdirs:
            continue
        if set(s.name for s in subdirs) >= {'train'}:
            return candidate
        sample = list(subdirs[0].iterdir())[:10]
        if any(f.suffix.lower() in IMG_EXTS for f in sample):
            return candidate
    raise RuntimeError(f'Cannot find data dir under {root}')

DATA_DIR = find_data_dir(EXTRACT_DIR)
print(f'DATA_DIR = {DATA_DIR}')

In [ ]:
# ── Helper: stream subprocess output live ─────────────────────────────────
import subprocess, sys

def run_streaming(cmd: list) -> int:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

## Train CustomCNN (baseline)
Trains from scratch ~30 epochs. Expect ~70%+ accuracy. Runtime: ~15 min on T4.

In [ ]:
rc = run_streaming([
    sys.executable, 'src/train.py',
    '--model',       'custom_cnn',
    '--data-dir',    str(DATA_DIR),
    '--epochs',      '30',
    '--batch-size',  '64',
    '--lr',          '1e-3',
    '--num-workers', '2',
])
print('\nCustomCNN exit code:', rc)

## Train MobileNetV2 (transfer learning)
Backbone frozen for first 5 epochs, then unfrozen at lr=1e-4. Target: ≥88% accuracy. Runtime: ~20 min on T4.

In [ ]:
rc = run_streaming([
    sys.executable, 'src/train.py',
    '--model',       'mobilenet_v2',
    '--data-dir',    str(DATA_DIR),
    '--epochs',      '25',
    '--batch-size',  '64',
    '--lr',          '1e-3',
    '--num-workers', '2',
])
print('\nMobileNetV2 exit code:', rc)

## Evaluate both models

In [ ]:
for model in ['custom_cnn', 'mobilenet_v2']:
    print(f'\n{"="*50}')
    print(f'Evaluating {model}...')
    run_streaming([
        sys.executable, 'src/evaluate.py',
        '--model',       model,
        '--data-dir',    str(DATA_DIR),
        '--num-workers', '2',
    ])

In [ ]:
# ── Compare models ────────────────────────────────────────────────────────
run_streaming([sys.executable, 'scripts/compare_models.py'])

In [ ]:
# ── Show training curves and confusion matrices ───────────────────────────
from IPython.display import Image, display
from pathlib import Path

for model in ['custom_cnn', 'mobilenet_v2']:
    for fig_name in [f'{model}_curves.png', f'{model}_confusion_matrix.png']:
        p = Path(f'outputs/figures/{fig_name}')
        if p.exists():
            print(f'\n{fig_name}:')
            display(Image(str(p)))

In [ ]:
# ── Save all outputs to Google Drive ─────────────────────────────────────
import shutil
from pathlib import Path

drive_out = Path('/content/drive/MyDrive/garbage-cnn-outputs')
drive_out.mkdir(parents=True, exist_ok=True)

for folder in ['outputs/figures', 'outputs/metrics', 'outputs/checkpoints']:
    src = Path(folder)
    if src.exists():
        dst = drive_out / src.name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved {folder} → {dst}')

print('\nAll outputs saved to Google Drive.')